In [2]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [3]:
# Clone the exact rail_ai repository
!git clone https://github.com/Mayank-Singh-X1/rail_ai.git
import os, sys
for path in ['/content/rail_ai/railway-surveillance-ai', '/content/rail_ai', 'railway-surveillance-ai', '.']:
    if os.path.exists(path) and path not in sys.path:
        sys.path.insert(0, path)
        try:
            os.chdir(path)
        except Exception:
            pass
print(f'Working directory: {os.getcwd()}')


Cloning into 'railway-surveillance-ai'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 30 (delta 0), reused 30 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), done.


In [4]:
!ls


config	   docs     modules	 README.md	   tests
dashboard  main.py  pipeline.py  requirements.txt  utils


In [5]:
# ============================================
# CELL 1: Install all dependencies
# ============================================
!pip install ultralytics insightface onnxruntime-gpu opencv-python-headless
!pip install deep-sort-realtime
!pip install streamlit gradio
!pip install firebase-admin twilio
!pip install scipy scikit-learn
!pip install lap  # for DeepSORT

# For CSRNet (crowd counting)
!git clone https://github.com/leeyeehoo/CSRNet-pytorch.git

# Verify GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.8/249.8 MB 5.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 15.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 25.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 59.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 82.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
import os
print(f"Current Directory: {os.getcwd()}")

Current Directory: /content/railway-surveillance-ai


In [ ]:
import os, sys
for path in ['/content/rail_ai/railway-surveillance-ai', '/content/rail_ai', 'railway-surveillance-ai', '.']:
    if os.path.exists(path) and path not in sys.path:
        sys.path.insert(0, path)
        try:
            os.chdir(path)
        except Exception:
            pass
print(f'Corrected Directory: {os.getcwd()}')


Corrected Directory: /content/railway-surveillance-ai


In [ ]:
# ============================================
# CELL 2: Initialize all models (PyTorch GPU)
# ============================================
import time
from collections import defaultdict, deque
import cv2
import insightface
import numpy as np
import torch
from insightface.app import FaceAnalysis
from ultralytics import YOLO


class RailwaySurveillanceSystem:

  def __init__(self):
    print("🚂 Initializing Railway Surveillance System...")

    # Determine execution provider
    self.device = 0 if torch.cuda.is_available() else "cpu"
    providers = (
        ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if torch.cuda.is_available()
        else ["CPUExecutionProvider"]
    )

    # ---- YOLO Models (Native PyTorch .pt) ----
    self.yolo_detect = YOLO("yolo11n.pt")
    print(f"✅ YOLO Detection model loaded on {self.device}")

    self.yolo_pose = YOLO("yolo11n-pose.pt")
    print(f"✅ YOLO Pose model loaded on {self.device}")

    # ---- InsightFace ----
    self.face_app = FaceAnalysis(name="buffalo_l", providers=providers)
    self.face_app.prepare(
        ctx_id=0 if torch.cuda.is_available() else -1, det_size=(640, 640)
    )
    print(f"✅ InsightFace model initialized (Providers: {providers})")

    # ---- Databases ----
    self.criminal_db = {}
    self.worker_db = {}
    self.worker_attendance = defaultdict(list)

    # ---- Tracking & Analytics ----
    self.track_history = defaultdict(lambda: deque(maxlen=50))
    self.crowd_history = deque(maxlen=300)

    # ---- Zones & Alerts ----
    self.zones = {}
    self.alerts = []

    print("🎉 System initialized successfully on GPU!")

  def _extract_normalized_embedding(self, image_path):
    img = cv2.imread(image_path)
    if img is None:
      print(f"❌ Could not read image at path: {image_path}")
      return None

    faces = self.face_app.get(img)
    if len(faces) > 0:
      embedding = faces[0].embedding
      normalized_embedding = embedding / np.linalg.norm(embedding)
      return normalized_embedding.astype(np.float32)

    return None

  def add_criminal_to_db(self, name, image_path):
    embedding = self._extract_normalized_embedding(image_path)
    if embedding is not None:
      self.criminal_db[name] = embedding
      print(f"✅ Criminal '{name}' added to vector database.")
      return True
    print(f"❌ No face detected in criminal image: {image_path}")
    return False

  def add_worker_to_db(self, name, image_path):
    embedding = self._extract_normalized_embedding(image_path)
    if embedding is not None:
      self.worker_db[name] = embedding
      print(f"✅ Worker '{name}' added to vector database.")
      return True
    print(f"❌ No face detected in worker image: {image_path}")
    return False

  def add_zone(self, zone_name, polygon_points, zone_type="restricted"):
    self.zones[zone_name] = {
        "polygon": np.array(polygon_points, dtype=np.int32),
        "type": zone_type,
    }
    print(f"✅ Zone '{zone_name}' ({zone_type}) configured.")


# Re-instantiate system object
system = RailwaySurveillanceSystem()

🚂 Initializing Railway Surveillance System...
✅ YOLO Detection model loaded on 0
✅ YOLO Pose model loaded on 0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/mode

In [ ]:
# ============================================
# CELL 2.1: Quick System Verification Test
# ============================================

# 1. Register a dummy restricted zone (e.g., Track/Platform boundary)
system.add_zone(
    zone_name="track_safety_zone",
    polygon_points=[[100, 400], [500, 400], [500, 600], [100, 600]],
    zone_type="restricted"
)

# 2. Check loaded models & GPU VRAM allocation
print(f"Tracking History initialized for {len(system.track_history)} objects.")
print(f"Zones configured: {list(system.zones.keys())}")

✅ Zone 'track_safety_zone' (restricted) configured.
Tracking History initialized for 0 objects.
Zones configured: ['track_safety_zone']


In [ ]:
# ============================================
# CELL 3: Crowd Analysis Module (YOLO11 Ready)
# ============================================
import time
from collections import Counter
import cv2
import numpy as np
from scipy.spatial import distance


class CrowdAnalyzer:

  def __init__(self, system):
    self.system = system
    self.density_thresholds = {
        "low": 20,
        "medium": 50,
        "high": 100,
        "critical": 200,
    }

  def count_people(self, frame):
    """Count people in real-time using YOLO11 detection model."""
    results = self.system.yolo_detect(
        frame,
        classes=[0],  # Class 0 = person in COCO dataset
        conf=0.3,
        verbose=False,
        device=self.system.device,
    )

    detections = results[0].boxes
    count = len(detections)

    # Store time-series entry for dynamic crowd trend analysis
    self.system.crowd_history.append({"count": count, "timestamp": time.time()})

    return count, detections

  def generate_density_heatmap(self, frame, detections):
    """Generate Gaussian crowd density heatmap overlaid on frame."""
    heatmap = np.zeros(frame.shape[:2], dtype=np.float32)

    for box in detections:
      x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
      cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

      # Create Gaussian point at detected person center
      cv2.circle(heatmap, (cx, cy), 50, 1, -1)

    # Apply Gaussian blur smoothing
    heatmap = cv2.GaussianBlur(heatmap, (99, 99), 0)

    # Normalize values to standard uint8 scale [0, 255]
    if heatmap.max() > 0:
      heatmap = (heatmap / heatmap.max() * 255).astype(np.uint8)

    # Render colormap overlay
    heatmap_colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(frame, 0.6, heatmap_colored, 0.4, 0)

    return overlay, heatmap

  def get_crowd_level(self, count):
    """Classify station crowd density state and assign status color."""
    if count < self.density_thresholds["low"]:
      return "LOW", (0, 255, 0)  # Green
    elif count < self.density_thresholds["medium"]:
      return "MEDIUM", (0, 255, 255)  # Yellow
    elif count < self.density_thresholds["high"]:
      return "HIGH", (0, 165, 255)  # Orange
    else:
      return "CRITICAL", (0, 0, 255)  # Red

  def get_zone_wise_count(self, frame, detections):
    """Evaluate occupant density across configured station polygons."""
    zone_counts = {}

    for zone_name, zone_info in self.system.zones.items():
      count = 0
      polygon = zone_info["polygon"]

      for box in detections:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

        # Verify point containment inside defined zone
        if cv2.pointPolygonTest(polygon, (float(cx), float(cy)), False) >= 0:
          count += 1

      zone_counts[zone_name] = count

    return zone_counts

  def predict_crowd_trend(self):
    """Forecast short-term crowd inflow/outflow trajectory."""
    if len(self.system.crowd_history) < 10:
      return "INSUFFICIENT DATA"

    recent = [h["count"] for h in list(self.system.crowd_history)[-10:]]
    older = [h["count"] for h in list(self.system.crowd_history)[-20:-10]]

    if not older:
      return "INSUFFICIENT DATA"

    recent_avg = np.mean(recent)
    older_avg = np.mean(older)

    if recent_avg > older_avg * 1.2:
      return "📈 INCREASING"
    elif recent_avg < older_avg * 0.8:
      return "📉 DECREASING"
    else:
      return "📊 STABLE"


# Instantiate crowd analyzer module
crowd_analyzer = CrowdAnalyzer(system)

In [ ]:
# ============================================
# CELL 4: Criminal Detection Module (Vector Accelerated)
# ============================================
import os
import time
import cv2
import numpy as np


class CriminalDetector:

  def __init__(self, system, similarity_threshold=0.45):
    self.system = system
    self.similarity_threshold = similarity_threshold
    self.alert_cooldown = {}
    self.COOLDOWN_SECONDS = 30

  def compute_similarity(self, embedding1, embedding2):
    """Fast dot product similarity for normalized embeddings."""
    return np.dot(embedding1, embedding2)

  def detect_criminals(self, frame):
    """Detect faces and match embeddings against database in parallel."""
    faces = self.system.face_app.get(frame)
    matches = []
    annotated_frame = frame.copy()

    if not faces or not self.system.criminal_db:
      return annotated_frame, matches

    # Extract names and normalized embedding matrix from DB for batch dot product
    db_names = list(self.system.criminal_db.keys())
    db_matrix = np.array(
        list(self.system.criminal_db.values()), dtype=np.float32
    )

    for face in faces:
      bbox = face.bbox.astype(int)

      # Extract & normalize live face embedding
      live_embedding = face.embedding / np.linalg.norm(face.embedding)

      # Vectorized matrix-vector multiplication (Compute similarity across DB at once)
      similarities = np.dot(db_matrix, live_embedding)
      best_idx = np.argmax(similarities)
      best_score = similarities[best_idx]
      best_match = db_names[best_idx]

      if best_score > self.similarity_threshold:
        current_time = time.time()

        # Handle alert cooldown to prevent notification spam
        if (
            best_match not in self.alert_cooldown
            or (current_time - self.alert_cooldown[best_match])
            > self.COOLDOWN_SECONDS
        ):

          gender_str = "Male" if face.gender == 1 else "Female"
          matches.append({
              "name": best_match,
              "score": float(best_score),
              "bbox": bbox,
              "age": int(face.age),
              "gender": gender_str,
          })

          self.alert_cooldown[best_match] = current_time

          # Push alert to system queue
          self.system.alerts.append({
              "type": "🚨 CRIMINAL DETECTED",
              "name": best_match,
              "confidence": f"{best_score:.2f}",
              "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
              "location": "Platform Camera 1",
          })

        # Render RED alert box for identified suspect
        cv2.rectangle(
            annotated_frame,
            (bbox[0], bbox[1]),
            (bbox[2], bbox[3]),
            (0, 0, 255),
            3,
        )
        label = f"⚠️ SUSPECT: {best_match} ({best_score:.2f})"
        cv2.putText(
            annotated_frame,
            label,
            (bbox[0], bbox[1] - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2,
        )
      else:
        # Render GREEN bounding box for normal passengers
        cv2.rectangle(
            annotated_frame,
            (bbox[0], bbox[1]),
            (bbox[2], bbox[3]),
            (0, 255, 0),
            2,
        )
        info = f"Age:{int(face.age)} {'M' if face.gender==1 else 'F'}"
        cv2.putText(
            annotated_frame,
            info,
            (bbox[0], bbox[1] - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1,
        )

    return annotated_frame, matches

  def bulk_register_criminals(self, image_folder):
    """Batch process criminal mugshots from a target folder into DB."""
    if not os.path.exists(image_folder):
      print(f"⚠️ Image directory '{image_folder}' does not exist.")
      return

    for filename in os.listdir(image_folder):
      if filename.endswith((".jpg", ".png", ".jpeg")):
        name = os.path.splitext(filename)[0]
        filepath = os.path.join(image_folder, filename)
        self.system.add_criminal_to_db(name, filepath)


# Instantiate Criminal Detector module
criminal_detector = CriminalDetector(system)

In [ ]:
# ============================================
# CELL 5: Anomaly Detection Module (YOLO11 Pose)
# ============================================
import time
from collections import defaultdict, deque
import cv2
import numpy as np


class AnomalyDetector:

  def __init__(self, system):
    self.system = system
    self.pose_history = defaultdict(lambda: deque(maxlen=30))

    # COCO Pose keypoint indices
    self.KEYPOINTS = {
        "nose": 0,
        "left_eye": 1,
        "right_eye": 2,
        "left_ear": 3,
        "right_ear": 4,
        "left_shoulder": 5,
        "right_shoulder": 6,
        "left_elbow": 7,
        "right_elbow": 8,
        "left_wrist": 9,
        "right_wrist": 10,
        "left_hip": 11,
        "right_hip": 12,
        "left_knee": 13,
        "right_knee": 14,
        "left_ankle": 15,
        "right_ankle": 16,
    }

  def detect_anomalies(self, frame):
    """Detect falls, rapid violent movements, and track intrusions via YOLO11 pose."""
    results = self.system.yolo_pose(
        frame, verbose=False, device=self.system.device
    )
    anomalies = []
    annotated_frame = frame.copy()

    if results[0].keypoints is not None:
      keypoints_data = results[0].keypoints.data.cpu().numpy()
      boxes = results[0].boxes

      for i, (kps, box) in enumerate(zip(keypoints_data, boxes)):
        person_id = i  # Tracks individual pose index per frame

        # Store keypoint history for velocity tracking
        self.pose_history[person_id].append(kps)

        # ---- FALL DETECTION ----
        if self._detect_fall(kps):
          bbox = box.xyxy[0].cpu().numpy().astype(int)
          anomalies.append({
              "type": "🆘 FALL DETECTED",
              "person_id": person_id,
              "bbox": bbox,
              "severity": "HIGH",
          })
          cv2.rectangle(
              annotated_frame,
              (bbox[0], bbox[1]),
              (bbox[2], bbox[3]),
              (0, 0, 255),
              3,
          )
          cv2.putText(
              annotated_frame,
              "⚠️ FALL DETECTED",
              (bbox[0], bbox[1] - 10),
              cv2.FONT_HERSHEY_SIMPLEX,
              0.8,
              (0, 0, 255),
              2,
          )

        # ---- FIGHTING/VIOLENCE DETECTION ----
        if self._detect_violence(kps, person_id):
          bbox = box.xyxy[0].cpu().numpy().astype(int)
          anomalies.append({
              "type": "👊 VIOLENCE DETECTED",
              "person_id": person_id,
              "bbox": bbox,
              "severity": "CRITICAL",
          })
          cv2.rectangle(
              annotated_frame,
              (bbox[0], bbox[1]),
              (bbox[2], bbox[3]),
              (255, 0, 0),
              3,
          )
          cv2.putText(
              annotated_frame,
              "👊 VIOLENCE",
              (bbox[0], bbox[1] - 10),
              cv2.FONT_HERSHEY_SIMPLEX,
              0.8,
              (255, 0, 0),
              2,
          )

        # ---- TRACK / ZONE INTRUSION ----
        # Calculate torso midpoint
        if (
            kps[self.KEYPOINTS["left_hip"]][2] > 0.3
            and kps[self.KEYPOINTS["right_hip"]][2] > 0.3
        ):
          cx = int(
              (
                  kps[self.KEYPOINTS["left_hip"]][0]
                  + kps[self.KEYPOINTS["right_hip"]][0]
              )
              / 2
          )
          cy = int(
              (
                  kps[self.KEYPOINTS["left_hip"]][1]
                  + kps[self.KEYPOINTS["right_hip"]][1]
              )
              / 2
          )

          for zone_name, zone_info in self.system.zones.items():
            if zone_info["type"] == "restricted":
              if (
                  cv2.pointPolygonTest(
                      zone_info["polygon"], (float(cx), float(cy)), False
                  )
                  >= 0
              ):
                anomalies.append({
                    "type": f"🚫 INTRUSION: {zone_name}",
                    "person_id": person_id,
                    "severity": "CRITICAL",
                })

    # Log anomalies to global alert queue
    for anomaly in anomalies:
      self.system.alerts.append(
          {**anomaly, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
      )

    return annotated_frame, anomalies

  def _detect_fall(self, keypoints):
    """Detect sudden collapse by measuring torso-to-ankle aspect ratios."""
    left_hip = keypoints[self.KEYPOINTS["left_hip"]]
    right_hip = keypoints[self.KEYPOINTS["right_hip"]]
    left_ankle = keypoints[self.KEYPOINTS["left_ankle"]]
    right_ankle = keypoints[self.KEYPOINTS["right_ankle"]]
    nose = keypoints[self.KEYPOINTS["nose"]]

    # Require minimum detection confidence
    if left_hip[2] < 0.3 or right_hip[2] < 0.3:
      return False

    hip_y = (left_hip[1] + right_hip[1]) / 2
    ankle_y = (
        (left_ankle[1] + right_ankle[1]) / 2
        if (left_ankle[2] > 0.3 and right_ankle[2] > 0.3)
        else hip_y + 10
    )

    vertical_diff = ankle_y - hip_y

    shoulder_l = keypoints[self.KEYPOINTS["left_shoulder"]]
    shoulder_r = keypoints[self.KEYPOINTS["right_shoulder"]]

    body_height = abs(nose[1] - ankle_y) if nose[2] > 0.3 else 0
    body_width = (
        abs(shoulder_l[0] - shoulder_r[0])
        if (shoulder_l[2] > 0.3 and shoulder_r[2] > 0.3)
        else 0
    )

    # Trigger fall alert if body orientation leans horizontal
    if body_height > 0 and body_width > 0:
      aspect_ratio = body_width / (body_height + 1e-6)
      if aspect_ratio > 1.4 or vertical_diff < 25:
        return True

    return False

  def _detect_violence(self, keypoints, person_id):
    """Detect fighting gestures via wrist movement velocity and position relative to shoulders."""
    if len(self.pose_history[person_id]) < 5:
      return False

    recent_poses = list(self.pose_history[person_id])[-5:]

    left_wrist_positions = [
        p[self.KEYPOINTS["left_wrist"]][:2]
        for p in recent_poses
        if p[self.KEYPOINTS["left_wrist"]][2] > 0.3
    ]
    right_wrist_positions = [
        p[self.KEYPOINTS["right_wrist"]][:2]
        for p in recent_poses
        if p[self.KEYPOINTS["right_wrist"]][2] > 0.3
    ]

    def compute_velocity(positions):
      if len(positions) < 3:
        return 0
      velocities = [
          np.sqrt(
              (positions[i][0] - positions[i - 1][0]) ** 2
              + (positions[i][1] - positions[i - 1][1]) ** 2
          )
          for i in range(1, len(positions))
      ]
      return np.mean(velocities) if velocities else 0

    left_vel = compute_velocity(left_wrist_positions)
    right_vel = compute_velocity(right_wrist_positions)

    # Check if wrists extend above shoulder plane
    hands_raised = False
    if (
        keypoints[self.KEYPOINTS["left_wrist"]][2] > 0.3
        and keypoints[self.KEYPOINTS["left_shoulder"]][2] > 0.3
    ):
      if (
          keypoints[self.KEYPOINTS["left_wrist"]][1]
          < keypoints[self.KEYPOINTS["left_shoulder"]][1]
      ):
        hands_raised = True

    if (
        keypoints[self.KEYPOINTS["right_wrist"]][2] > 0.3
        and keypoints[self.KEYPOINTS["right_shoulder"]][2] > 0.3
    ):
      if (
          keypoints[self.KEYPOINTS["right_wrist"]][1]
          < keypoints[self.KEYPOINTS["right_shoulder"]][1]
      ):
        hands_raised = True

    VELOCITY_THRESHOLD = 45  # Velocity threshold for rapid movement
    return (
        left_vel > VELOCITY_THRESHOLD or right_vel > VELOCITY_THRESHOLD
    ) and hands_raised


# Instantiate Anomaly Detector module
anomaly_detector = AnomalyDetector(system)

In [ ]:
# ============================================
# CELL 6: Cleanliness Detection Module (YOLO11)
# ============================================
import cv2
import numpy as np
from ultralytics import YOLO


class CleanlinessMonitor:
  """Automated cleanliness scoring and litter detection using YOLO11."""

  def __init__(self, system):
    self.system = system

    # COCO object classes indicative of platform litter for default demo mode
    self.dirty_classes = {
        39: "bottle",
        40: "wine glass",
        41: "cup",
        73: "paper/book",
        76: "trash item",
    }

    self.custom_model = None

  def train_custom_model(self, dataset_yaml_path):
    """Fine-tune YOLO11 on custom railway waste/litter datasets."""
    model = YOLO("yolo11n.pt")
    results = model.train(
        data=dataset_yaml_path,
        epochs=50,
        imgsz=640,
        batch=16,
        name="railway_cleanliness",
        patience=10,
        augment=True,
        mosaic=1.0,
        mixup=0.1,
        device=self.system.device,
    )
    self.custom_model = YOLO("runs/detect/railway_cleanliness/weights/best.pt")
    return results

  def detect_uncleanliness(self, frame):
    """Identify waste items and render bounding box overlays."""
    detections = []
    annotated_frame = frame.copy()

    # Route inference through fine-tuned model or fall back to system detector
    if self.custom_model:
      results = self.custom_model(
          frame, conf=0.3, verbose=False, device=self.system.device
      )
    else:
      results = self.system.yolo_detect(
          frame, conf=0.3, verbose=False, device=self.system.device
      )

    for box in results[0].boxes:
      cls_id = int(box.cls[0])

      if cls_id in self.dirty_classes or self.custom_model:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        conf = float(box.conf[0])

        if self.custom_model:
          label = results[0].names[cls_id]
        else:
          label = self.dirty_classes.get(cls_id, "litter")

        detections.append(
            {"type": label, "bbox": (x1, y1, x2, y2), "confidence": conf}
        )

        # Draw ORANGE bounding box for uncleanliness detections
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 165, 255), 2)
        cv2.putText(
            annotated_frame,
            f"🗑️ {label} ({conf:.2f})",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 165, 255),
            2,
        )

    return annotated_frame, detections

  def get_cleanliness_score(self, frame):
    """Calculate zone cleanliness score on a scale of 0 to 100."""
    _, detections = self.detect_uncleanliness(frame)

    # Base score (100 = perfectly clean)
    score = 100.0

    # Deduct points based on detected waste volume and confidence
    deduction_per_item = 8.0
    score -= len(detections) * deduction_per_item

    if detections:
      avg_conf = np.mean([d["confidence"] for d in detections])
      score -= avg_conf * 5.0

    return float(max(0.0, min(100.0, score)))


# Instantiate Cleanliness Monitor module
cleanliness_monitor = CleanlinessMonitor(system)

In [ ]:
# ============================================
# CELL 7: Worker Monitoring Module (Vector Accelerated)
# ============================================
import time
from collections import defaultdict
import cv2
import numpy as np


class WorkerMonitor:

  def __init__(self, system):
    self.system = system
    self.attendance_log = defaultdict(list)
    self.zone_time_tracker = defaultdict(lambda: defaultdict(float))
    self.last_seen = {}

  def check_attendance(self, frame):
    """Detect workers and maintain real-time attendance logs."""
    faces = self.system.face_app.get(frame)
    present_workers = []
    annotated_frame = frame.copy()

    if not faces or not self.system.worker_db:
      all_workers = set(self.system.worker_db.keys())
      return annotated_frame, [], list(all_workers)

    # Convert worker DB into matrix for batch vector comparison
    db_names = list(self.system.worker_db.keys())
    db_matrix = np.array(
        list(self.system.worker_db.values()), dtype=np.float32
    )

    for face in faces:
      bbox = face.bbox.astype(int)

      # Extract L2-normalized embedding
      live_embedding = face.embedding / np.linalg.norm(face.embedding)

      # Matrix multiplication for cosine similarity
      similarities = np.dot(db_matrix, live_embedding)
      best_idx = np.argmax(similarities)
      best_score = similarities[best_idx]
      best_match = db_names[best_idx]

      if best_score > 0.4:
        present_workers.append(best_match)
        current_timestamp = time.strftime("%Y-%m-%d %H:%M:%S")

        # Record attendance once every hour (3600s)
        if (
            best_match not in self.last_seen
            or (time.time() - self.last_seen.get(best_match, 0)) > 3600
        ):
          self.attendance_log[best_match].append(current_timestamp)

        self.last_seen[best_match] = time.time()

        # Track spatial zone presence
        self.track_zone_presence(best_match, bbox)

        # Draw CYAN/BLUE bounding box for recognized station staff
        cv2.rectangle(
            annotated_frame,
            (bbox[0], bbox[1]),
            (bbox[2], bbox[3]),
            (255, 200, 0),
            2,
        )
        cv2.putText(
            annotated_frame,
            f"👷 {best_match} ({best_score:.2f})",
            (bbox[0], bbox[1] - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 200, 0),
            2,
        )

    # Determine absent staff
    all_workers = set(self.system.worker_db.keys())
    absent_workers = all_workers - set(present_workers)

    return annotated_frame, present_workers, list(absent_workers)

  def track_zone_presence(self, worker_name, worker_bbox):
    """Track time spent by staff inside designated station zones."""
    cx = (worker_bbox[0] + worker_bbox[2]) // 2
    cy = (worker_bbox[1] + worker_bbox[3]) // 2

    for zone_name, zone_info in self.system.zones.items():
      if (
          cv2.pointPolygonTest(
              zone_info["polygon"], (float(cx), float(cy)), False
          )
          >= 0
      ):
        # Accumulate estimated duration in seconds
        self.zone_time_tracker[worker_name][zone_name] += 0.033

  def get_attendance_report(self):
    """Summarize staff presence and zone activity."""
    report = {}
    for name in self.system.worker_db.keys():
      report[name] = {
          "status": "Present" if name in self.last_seen else "Absent",
          "check_in_times": self.attendance_log.get(name, []),
          "zones_visited_seconds": {
              k: round(v, 1)
              for k, v in self.zone_time_tracker.get(name, {}).items()
          },
      }
    return report


# Instantiate Worker Monitor module
worker_monitor = WorkerMonitor(system)

In [ ]:
# ============================================
# CELL 8: ByteTrack Tracking Integration (YOLO11 Native)
# ============================================
from collections import defaultdict, deque
import cv2
import numpy as np


class PersonTracker:

  def __init__(self, system):
    self.system = system
    self.track_history = defaultdict(lambda: deque(maxlen=100))
    self.line_crossings = defaultdict(int)

  def track_persons(self, frame):
    """Detect and track persons instantly using YOLO11 + ByteTrack."""
    # Native YOLO11 tracking via ByteTrack
    results = self.system.yolo_detect.track(
        frame,
        persist=True,
        classes=[0],  # class 0 = person
        conf=0.3,
        tracker="bytetrack.yaml",
        verbose=False,
        device=self.system.device,
    )

    annotated_frame = frame.copy()
    active_tracks = 0

    if results[0].boxes is not None and results[0].boxes.id is not None:
      boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
      track_ids = results[0].boxes.id.cpu().numpy().astype(int)

      active_tracks = len(track_ids)

      for box, track_id in zip(boxes, track_ids):
        x1, y1, x2, y2 = box

        # Store path history for trajectory visualization
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        self.track_history[track_id].append((cx, cy))

        # Draw bounding box & persistent ID
        cv2.rectangle(
            annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2
        )
        cv2.putText(
            annotated_frame,
            f"ID: {track_id}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2,
        )

        # Render movement trail
        history = list(self.track_history[track_id])
        for j in range(1, len(history)):
          thickness = int(np.sqrt(64 / float(j + 1)) * 2)
          cv2.line(
              annotated_frame,
              history[j - 1],
              history[j],
              (230, 230, 0),
              thickness,
          )

    return annotated_frame, active_tracks

  def setup_line_counter(self, line_start, line_end):
    """Configure virtual line for entry/exit counting."""
    self.count_line = (line_start, line_end)

  def count_line_crossings(self, frame):
    """Render counting line and aggregate crossings on frame."""
    annotated_frame, _ = self.track_persons(frame)

    if hasattr(self, "count_line"):
      pt1, pt2 = self.count_line
      cv2.line(annotated_frame, pt1, pt2, (0, 0, 255), 3)

      cv2.putText(
          annotated_frame,
          f"Crossings: {sum(self.line_crossings.values())}",
          (pt1[0], pt1[1] - 20),
          cv2.FONT_HERSHEY_SIMPLEX,
          1,
          (0, 0, 255),
          2,
      )

    return annotated_frame


# Instantiate Person Tracker module
person_tracker = PersonTracker(system)

In [ ]:
import sys
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

"""
Unified Pipeline and System Initializer for Railway Surveillance AI
"""
import os
import json
import time
from collections import defaultdict, deque
import cv2
import numpy as np
import torch

try:
    from ultralytics import YOLO
except ImportError:
    class MockResult:
        def __init__(self):
            self.boxes = []
            self.names = {0: "person", 39: "bottle"}
            self.keypoints = type("KP", (), {"xy": torch.zeros((1, 17, 2)), "data": torch.zeros((1, 17, 3))})()
            self.orig_shape = (720, 1280)

    class MockYOLO:
        def __init__(self, *args, **kwargs):
            self.names = {0: "person", 39: "bottle"}
        def __call__(self, frame, *args, **kwargs):
            return [MockResult()]
        def track(self, frame, *args, **kwargs):
            return [MockResult()]
        def train(self, *args, **kwargs):
            return {}

    YOLO = MockYOLO

try:
    from insightface.app import FaceAnalysis
    HAS_INSIGHTFACE = True
except ImportError:
    class MockFace:
        def __init__(self):
            self.embedding = np.ones(512, dtype=np.float32)
            self.bbox = np.array([50, 50, 200, 200])

    class MockFaceAnalysis:
        def __init__(self, *args, **kwargs): pass
        def prepare(self, *args, **kwargs): pass
        def get(self, img):
            return [MockFace()]

    FaceAnalysis = MockFaceAnalysis
    HAS_INSIGHTFACE = False

from modules import (
    CrowdAnalyzer,
    CriminalDetector,
    AnomalyDetector,
    CleanlinessMonitor,
    WorkerMonitor,
    PersonTracker,
    AlertSystem
)


class RailwaySurveillanceSystem:

  def __init__(self):
    print("🚂 Initializing Railway Surveillance System...")

    # Determine execution provider
    self.device = 0 if torch.cuda.is_available() else "cpu"
    providers = (
        ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if torch.cuda.is_available()
        else ["CPUExecutionProvider"]
    )

    # ---- YOLO Models (Native PyTorch .pt) ----
    try:
        self.yolo_detect = YOLO("yolo11n.pt")
        print(f"✅ YOLO Detection model loaded on {self.device}")
    except Exception as e:
        print(f"⚠️ YOLO Detection fallback: {e}")
        self.yolo_detect = YOLO()

    self.yolo = self.yolo_detect

    try:
        self.yolo_pose = YOLO("yolo11n-pose.pt")
        print(f"✅ YOLO Pose model loaded on {self.device}")
    except Exception as e:
        print(f"⚠️ YOLO Pose fallback: {e}")
        self.yolo_pose = YOLO()

    # ---- InsightFace ----
    try:
        self.face_app = FaceAnalysis(name="buffalo_l", providers=providers)
        self.face_app.prepare(
            ctx_id=0 if torch.cuda.is_available() else -1, det_size=(640, 640)
        )
        print(f"✅ InsightFace model initialized (Providers: {providers})")
    except Exception as e:
        print(f"⚠️ InsightFace fallback: {e}")
        self.face_app = FaceAnalysis()

    # ---- Databases ----
    self.criminal_db = {}
    self.worker_db = {}
    self.worker_attendance = defaultdict(list)

    # ---- Tracking & Analytics ----
    self.track_history = defaultdict(lambda: deque(maxlen=50))
    self.crowd_history = deque(maxlen=300)

    # ---- Zones & Alerts ----
    self.zones = {}
    self.alerts = []

    print("🎉 System initialized successfully on GPU/CPU!")

  def _extract_normalized_embedding(self, image_path):
    img = cv2.imread(image_path)
    if img is None:
      print(f"❌ Could not read image at path: {image_path}")
      return None

    faces = self.face_app.get(img)
    if len(faces) > 0:
      embedding = faces[0].embedding
      normalized_embedding = embedding / np.linalg.norm(embedding)
      return normalized_embedding.astype(np.float32)

    return None

  def add_criminal_to_db(self, name, image_path):
    embedding = self._extract_normalized_embedding(image_path)
    if embedding is not None:
      self.criminal_db[name] = embedding
      print(f"✅ Criminal '{name}' added to vector database.")
      return True
    print(f"❌ No face detected in criminal image: {image_path}")
    return False

  def add_worker_to_db(self, name, image_path):
    embedding = self._extract_normalized_embedding(image_path)
    if embedding is not None:
      self.worker_db[name] = embedding
      print(f"✅ Worker '{name}' added to vector database.")
      return True
    print(f"❌ No face detected in worker image: {image_path}")
    return False

  def add_zone(self, zone_name, polygon_points, zone_type="restricted"):
    self.zones[zone_name] = {
        "polygon": np.array(polygon_points, dtype=np.int32),
        "type": zone_type,
    }
    print(f"✅ Zone '{zone_name}' ({zone_type}) configured.")


# Global system instances
system = RailwaySurveillanceSystem()
crowd_analyzer = CrowdAnalyzer(system)
criminal_detector = CriminalDetector(system)
anomaly_detector = AnomalyDetector(system)
cleanliness_monitor = CleanlinessMonitor(system)
worker_monitor = WorkerMonitor(system)
person_tracker = PersonTracker(system)
alert_system = AlertSystem()


class UnifiedPipeline:

  def __init__(self):
    self.system = system
    self.crowd_analyzer = crowd_analyzer
    self.criminal_detector = criminal_detector
    self.anomaly_detector = anomaly_detector
    self.cleanliness_monitor = cleanliness_monitor
    self.worker_monitor = worker_monitor
    self.person_tracker = person_tracker

    self.frame_count = 0
    self.fps_history = deque(maxlen=30)

    # State caches to prevent UI flickering during frame-skipping
    self.cached_criminals = []
    self.cached_anomalies = []
    self.cached_cleanliness_score = 100.0
    self.cached_workers_present = []
    self.cached_workers_absent = []

  def process_frame(
      self,
      frame,
      enable_crowd=True,
      enable_criminal=True,
      enable_anomaly=True,
      enable_cleanliness=True,
      enable_tracking=True,
      enable_worker=True,
  ):
    """Process a single frame through enabled modules efficiently."""

    start_time = time.time()
    self.frame_count += 1

    annotated = frame.copy()

    results = {
        "frame": None,
        "crowd_count": 0,
        "crowd_level": "N/A",
        "criminals_found": [],
        "anomalies": [],
        "cleanliness_score": self.cached_cleanliness_score,
        "workers_present": [],
        "workers_absent": [],
        "tracked_persons": 0,
        "alerts": [],
    }

    # ---- 1. TRACKING & CROWD (Every Frame - Combined YOLO11 Pass) ----
    if enable_tracking:
      annotated, active_count = self.person_tracker.track_persons(annotated)
      results["tracked_persons"] = active_count
      results["crowd_count"] = active_count
    elif enable_crowd:
      count, detections = self.crowd_analyzer.count_people(frame)
      results["crowd_count"] = count

    if enable_crowd:
      level, color = self.crowd_analyzer.get_crowd_level(
          results["crowd_count"]
      )
      results["crowd_level"] = level


    # ---- 2. CRIMINAL DETECTION (Every 5th Frame) ----
    if enable_criminal and (self.frame_count % 5 == 0):
      _, matches = self.criminal_detector.detect_criminals(frame)
      self.cached_criminals = matches

    results["criminals_found"] = self.cached_criminals

    # Render cached criminal alerts
    for match in self.cached_criminals:
      bbox = match["bbox"]
      cv2.rectangle(
          annotated, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 0, 255), 3
      )
      cv2.putText(
          annotated,
          f"🚨 {match['name']} ({match['score']:.2f})",
          (bbox[0], bbox[1] - 15),
          cv2.FONT_HERSHEY_SIMPLEX,
          0.7,
          (0, 0, 255),
          2,
      )

    # ---- 3. ANOMALY DETECTION (Every 3rd Frame) ----
    if enable_anomaly and (self.frame_count % 3 == 0):
      _, anomalies = self.anomaly_detector.detect_anomalies(frame)
      self.cached_anomalies = anomalies

    results["anomalies"] = self.cached_anomalies

    # ---- 4. CLEANLINESS MONITORING (Every 30th Frame) ----
    if enable_cleanliness and (self.frame_count % 30 == 0):
      self.cached_cleanliness_score = (
          self.cleanliness_monitor.get_cleanliness_score(frame)
      )

    results["cleanliness_score"] = self.cached_cleanliness_score

    # ---- 5. WORKER MONITORING (Every 10th Frame) ----
    if enable_worker and (self.frame_count % 10 == 0):
      _, present, absent = self.worker_monitor.check_attendance(frame)
      self.cached_workers_present = present
      self.cached_workers_absent = absent

    results["workers_present"] = self.cached_workers_present
    results["workers_absent"] = self.cached_workers_absent

    # ---- 6. FPS CALCULATION ----
    elapsed = time.time() - start_time
    fps = 1.0 / (elapsed + 1e-6)
    self.fps_history.append(fps)
    avg_fps = np.mean(self.fps_history)

    # ---- 7. OVERLAYS & HUD RENDERING ----
    from utils.visualization import draw_hud, draw_zones
    if hasattr(self.system, 'zones') and self.system.zones:
        draw_zones(annotated, self.system.zones)

    draw_hud(annotated, results)

    results["frame"] = annotated
    results["alerts"] = self.system.alerts[-10:]

    return results

  def _draw_status_bar(self, frame, results):
    """Fallback alias for HUD drawing."""
    from utils.visualization import draw_hud
    draw_hud(frame, results)



pipeline = UnifiedPipeline()


def process_video(
    video_path,
    output_path="output_surveillance.mp4",
    max_frames=200,
    target_width=1280,
    frame_stride=3,
):
    """Process a video file through the unified surveillance pipeline with fast downscaling and frame skipping."""
    if not os.path.exists(video_path):
        print(f"❌ Video not found at path: {video_path}")
        return None

    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    scale = target_width / orig_w
    target_height = int(orig_h * scale)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(
        output_path, fourcc, fps // frame_stride, (target_width, target_height)
    )

    frame_idx = 0
    processed_count = 0

    while cap.isOpened() and processed_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_stride == 0:
            frame = cv2.resize(
                frame, (target_width, target_height), interpolation=cv2.INTER_AREA
            )
            result = pipeline.process_frame(frame)
            out.write(result["frame"])
            processed_count += 1

        frame_idx += 1

    cap.release()
    out.release()
    print(f"🎬 Video processing complete! Saved to {output_path}")
    return output_path


def stream_live_camera(source="0", frame_stride=2, target_width=1280, target_height=720, max_frames=300):
    """
    Generator function for real-time live feed.
    Supports local webcams, RTSP streams, video files, and Colab simulation fallback.
    """
    if str(source).isdigit():
        source = int(source)

    cap = cv2.VideoCapture(source)
    is_valid_source = cap.isOpened()

    if not is_valid_source:
        print(f"⚠️ Physical camera not detected on cloud server ({source}). Switching to Live Railway CCTV Simulation Mode...")

    frame_idx = 0
    last_frame = None

    while frame_idx < max_frames:
        if is_valid_source:
            ret, frame = cap.read()
            if not ret:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0) # Loop video
                ret, frame = cap.read()
                if not ret:
                    break
        else:
            # Generate dynamic realistic Railway CCTV stream simulation
            frame = np.zeros((target_height, target_width, 3), dtype=np.uint8)
            frame[:] = (40, 45, 50) # Dark platform background

            # Platform edge and yellow safety line
            cv2.line(frame, (0, target_height - 180), (target_width, target_height - 180), (0, 215, 255), 4)
            cv2.line(frame, (0, target_height - 100), (target_width, target_height - 100), (80, 80, 80), 3)
            cv2.putText(frame, "PLATFORM 2 - CAUTION: STAY BEHIND YELLOW LINE", (40, target_height - 195),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 215, 255), 2)

            # Simulated moving passenger targets
            t = frame_idx * 0.08
            num_passengers = 6 + int(3 * np.sin(t * 0.5))
            for i in range(num_passengers):
                px = int((target_width * 0.15 + (i * 140) + np.sin(t + i) * 60) % (target_width - 100))
                py = int(220 + (i % 3) * 80 + np.cos(t * 0.8 + i) * 30)
                # Draw passenger representation
                cv2.circle(frame, (px + 25, py - 15), 18, (180, 200, 220), -1) # Head
                cv2.rectangle(frame, (px, py), (px + 50, py + 110), (140, 150, 160), -1) # Body
                cv2.putText(frame, "person", (px, py - 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

            time.sleep(0.04) # Simulate 25 FPS stream timing

        frame = cv2.resize(frame, (target_width, target_height), interpolation=cv2.INTER_AREA)

        if frame_idx % frame_stride == 0:
            result = pipeline.process_frame(frame)
            last_frame = result["frame"]

        frame_idx += 1
        yield last_frame if last_frame is not None else frame

    if is_valid_source:
        cap.release()


In [ ]:
# ============================================
# CELL 10: High-Speed Video Processing Pipeline
# ============================================
import os
import cv2


def process_video(
    video_path,
    output_path="output_surveillance.mp4",
    max_frames=200,  # Lowered default cap
    target_width=1280,  # Resolution downscaling
    frame_stride=3,  # Process AI every Nth frame (1 = every frame, 2 = every 2nd frame)
):
  """Process a video file through the unified surveillance pipeline with fast downscaling and frame skipping."""

  if not os.path.exists(video_path):
    print(f"❌ Input video file not found at: {video_path}")
    return None

  cap = cv2.VideoCapture(video_path)
  if not cap.isOpened():
    print(f"❌ Cannot open video: {video_path}")
    return None

  # Retrieve source properties
  fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
  orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

  # Calculate aspect-ratio preserved target dimensions
  aspect_ratio = orig_h / orig_w
  target_height = int(target_width * aspect_ratio)

  print(
      f"📹 Source: {orig_w}x{orig_h} | Target Processing Resolution:"
      f" {target_width}x{target_height} @ {fps} FPS"
  )
  print(
      f"⚡ Optimization: Max Frames = {max_frames} | Frame Stride ="
      f" {frame_stride}x speedup"
  )

  # Configure VideoWriter using avc1 / mp4v
  fourcc = cv2.VideoWriter_fourcc(*"avc1")
  out = cv2.VideoWriter(
      output_path, fourcc, fps, (target_width, target_height)
  )

  frame_count = 0
  all_results = []
  last_annotated_frame = None
  last_results_telemetry = None

  while cap.isOpened() and frame_count < max_frames:
    ret, frame = cap.read()
    if not ret:
      break

    # 1. Downscale frame before deep learning models run
    resized_frame = cv2.resize(
        frame, (target_width, target_height), interpolation=cv2.INTER_LINEAR
    )

    # 2. Frame-skipping logic for fast execution
    if frame_count % frame_stride == 0 or last_annotated_frame is None:
      # Heavy AI Model Pass
      results = pipeline.process_frame(resized_frame)
      annotated_frame = results["frame"]
      last_annotated_frame = annotated_frame

      # Construct telemetry
      last_results_telemetry = {
          "frame": frame_count,
          "crowd_count": results.get("crowd_count", 0),
          "crowd_level": results.get("crowd_level", "NORMAL"),
          "cleanliness_score": results.get("cleanliness_score", 100.0),
          "criminals": len(results.get("criminals_found", [])),
          "anomalies": len(results.get("anomalies", [])),
      }
    else:
      # Reuse cached annotated frame & telemetry
      annotated_frame = last_annotated_frame
      last_results_telemetry["frame"] = frame_count

    # Ensure shape safety before writing
    if (
        annotated_frame.shape[1] != target_width
        or annotated_frame.shape[0] != target_height
    ):
      annotated_frame = cv2.resize(
          annotated_frame, (target_width, target_height)
      )

    out.write(annotated_frame)
    all_results.append(last_results_telemetry)

    frame_count += 1
    if frame_count % 50 == 0:
      print(
          f"⏳ Processed {frame_count}/{min(max_frames, total_frames)} frames..."
      )

  cap.release()
  out.release()

  print(
      f"✅ Fast processing complete! Encoded {frame_count} frames.\n💾 Saved to:"
      f" {output_path}"
  )
  return all_results

In [ ]:
# ============================================
# REAL-TIME LIVE STREAMING & SIMULATION GENERATOR
# ============================================
import cv2
import time
import numpy as np

def stream_live_camera(source="0", frame_stride=2, target_width=1280, target_height=720, max_frames=300):
    if str(source).isdigit():
        source = int(source)

    cap = cv2.VideoCapture(source)
    is_valid_source = cap.isOpened()

    if not is_valid_source:
        print(f"⚠️ Cloud server has no physical camera attached. Running Live Railway CCTV Simulation Mode...")

    frame_idx = 0
    last_frame = None

    while frame_idx < max_frames:
        if is_valid_source:
            ret, frame = cap.read()
            if not ret:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                ret, frame = cap.read()
                if not ret:
                    break
        else:
            # Dynamic simulated Railway CCTV stream
            frame = np.zeros((target_height, target_width, 3), dtype=np.uint8)
            frame[:] = (40, 45, 50)
            cv2.line(frame, (0, target_height - 180), (target_width, target_height - 180), (0, 215, 255), 4)
            cv2.putText(frame, "PLATFORM 2 - CAUTION: STAY BEHIND YELLOW LINE", (40, target_height - 195),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 215, 255), 2)
            t = frame_idx * 0.08
            num_passengers = 6 + int(3 * np.sin(t * 0.5))
            for i in range(num_passengers):
                px = int((target_width * 0.15 + (i * 140) + np.sin(t + i) * 60) % (target_width - 100))
                py = int(220 + (i % 3) * 80 + np.cos(t * 0.8 + i) * 30)
                cv2.circle(frame, (px + 25, py - 15), 18, (180, 200, 220), -1)
                cv2.rectangle(frame, (px, py), (px + 50, py + 110), (140, 150, 160), -1)
                cv2.putText(frame, "person", (px, py - 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            time.sleep(0.04)

        frame = cv2.resize(frame, (target_width, target_height), interpolation=cv2.INTER_AREA)

        if frame_idx % frame_stride == 0:
            result = pipeline.process_frame(frame)
            last_frame = result["frame"]

        frame_idx += 1
        yield last_frame if last_frame is not None else frame

    if is_valid_source:
        cap.release()


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# results = process_video("/content/drive/MyDrive/test_video.mp4")

In [ ]:
# ============================================
# CELL 11: Interactive Dashboard (Gradio with Live Webcam & Stream)
# ============================================
import sys, os
for p in ['/content/rail_ai/railway-surveillance-ai', '/content/rail_ai', '/content/railway-surveillance-ai', '.', '..']:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)
from pipeline import RailwaySurveillanceSystem, UnifiedPipeline
try:
    from pipeline import system, pipeline, process_video, stream_live_camera
except Exception:
    system = RailwaySurveillanceSystem()
    pipeline = UnifiedPipeline()


/tmp/ipykernel_1939/2990534770.py:70: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="🚂 Indian Railways AI Surveillance System", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://05040fb23cb0ae3b19.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


❌ Cannot connect to live source: test_video.mp4
❌ Cannot connect to live source: 0
❌ Cannot connect to live source: 0
❌ Cannot connect to live source: 0
❌ Cannot connect to live source: 
❌ Cannot connect to live source: 0
❌ Cannot connect to live source: 0
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://05040fb23cb0ae3b19.gradio.live


In [ ]:
# ============================================
# CELL 12: Alert & Notification System
# ============================================
import json
import os
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import smtplib


class AlertSystem:

  def __init__(self):
    self.alert_log = []
    # Configure environment variables or app credentials
    self.email_config = {
        'smtp_server': os.getenv('SMTP_SERVER', 'smtp.gmail.com'),
        'smtp_port': int(os.getenv('SMTP_PORT', 587)),
        'sender_email': os.getenv('SENDER_EMAIL', 'your_email@gmail.com'),
        'sender_password': os.getenv('SENDER_PASSWORD', 'your_app_password'),
        'receiver_emails': ['security@railway.gov.in'],
    }

  def send_alert(self, alert_data):
    """Dispatch alerts across console, log files, and email based on severity."""
    if 'timestamp' not in alert_data:
      alert_data['timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    self.alert_log.append(alert_data)
    severity = alert_data.get('severity', 'LOW').upper()

    # Always log to console
    self._console_alert(alert_data)

    # File logging
    self._log_alert(alert_data)

    # Route critical threats via Email dispatch
    if severity in ['HIGH', 'CRITICAL']:
      self._email_alert(alert_data)

  def _console_alert(self, alert_data):
    """Display formatted notification in the terminal/console."""
    severity = alert_data.get('severity', 'UNKNOWN')
    icon = '🚨' if severity in ['HIGH', 'CRITICAL'] else '⚠️'

    print(f"\n{'='*50}")
    print(f"{icon} ALERT DETECTED: {alert_data.get('type', 'General Alert')}")
    print(f"📍 Location: {alert_data.get('location', 'Platform Stream 01')}")
    print(f"⏰ Timestamp: {alert_data.get('timestamp')}")
    print(f"⚡ Severity Level: {severity}")
    if 'name' in alert_data:
      print(f"👤 Target ID: {alert_data['name']}")
    if 'details' in alert_data:
      print(f"📝 Notes: {alert_data['details']}")
    print(f"{'='*50}\n")

  def _email_alert(self, alert_data):
    """Send formatted alert emails to security personnel."""
    # Skip processing if default template parameters are unconfigured
    if self.email_config['sender_email'] == 'your_email@gmail.com':
      print("📧 [Simulated Email] High-severity alert triggered!")
      return

    try:
      msg = MIMEMultipart()
      msg['From'] = self.email_config['sender_email']
      msg['To'] = ', '.join(self.email_config['receiver_emails'])
      msg['Subject'] = (
          f"🚨 Railway Security Alert: {alert_data.get('type', 'Incident')}"
      )

      body = f"""
            RAILWAY SURVEILLANCE AUTOMATED ALERT
            ------------------------------------
            Alert Type : {alert_data.get('type', 'Unknown')}
            Severity   : {alert_data.get('severity', 'HIGH')}
            Time       : {alert_data.get('timestamp')}
            Location   : {alert_data.get('location', 'Platform Stream')}
            
            Incident Details:
            {json.dumps(alert_data, indent=2, default=str)}
            """
      msg.attach(MIMEText(body, 'plain'))

      server = smtplib.SMTP(
          self.email_config['smtp_server'], self.email_config['smtp_port']
      )
      server.starttls()
      server.login(
          self.email_config['sender_email'],
          self.email_config['sender_password'],
      )
      server.sendmail(
          self.email_config['sender_email'],
          self.email_config['receiver_emails'],
          msg.as_string(),
      )
      server.quit()

      print("📧 Email alert dispatched successfully!")
    except Exception as e:
      print(f"❌ Failed to deliver email alert: {e}")

  def _log_alert(self, alert_data):
    """Append structured telemetry record to alerts_log.json."""
    try:
      with open('alerts_log.json', 'a') as f:
        f.write(json.dumps(alert_data, default=str) + '\n')
    except Exception as e:
      print(f"❌ Failed to log alert to disk: {e}")

  def get_alert_summary(self):
    """Summarize aggregated alerts by type and severity level."""
    summary = {
        'total_alerts': len(self.alert_log),
        'by_type': {},
        'by_severity': {},
    }

    for alert in self.alert_log:
      alert_type = alert.get('type', 'Unknown')
      severity = alert.get('severity', 'Unknown')

      summary['by_type'][alert_type] = (
          summary['by_type'].get(alert_type, 0) + 1
      )
      summary['by_severity'][severity] = (
          summary['by_severity'].get(severity, 0) + 1
      )

    return summary


# Instantiate Alert System module
alert_system = AlertSystem()

In [ ]:
# ============================================
# CELL 13: Analytics & Visualization
# ============================================
import time
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.use('Agg')


class AnalyticsDashboard:

  @staticmethod
  def plot_crowd_trend(crowd_history):
    """Plot crowd count over time."""
    fig, ax = plt.subplots(1, 1, figsize=(12, 4))

    counts = [
        h['count'] if isinstance(h, dict) else h for h in crowd_history
    ]
    if not counts:
      counts = [0]

    timestamps = list(range(len(counts)))

    ax.fill_between(timestamps, counts, alpha=0.3, color='blue')
    ax.plot(timestamps, counts, color='blue', linewidth=2)
    ax.set_xlabel('Time (frames)')
    ax.set_ylabel('People Count')
    ax.set_title('👥 Crowd Density Over Time')
    ax.grid(True, alpha=0.3)

    # Threshold lines
    ax.axhline(
        y=50, color='orange', linestyle='--', label='Medium Threshold', alpha=0.7
    )
    ax.axhline(
        y=100, color='red', linestyle='--', label='High Threshold', alpha=0.7
    )
    ax.legend()

    plt.tight_layout()
    output_filename = 'crowd_trend.png'
    plt.savefig(output_filename, dpi=150, bbox_inches='tight')
    plt.close()
    return output_filename

  @staticmethod
  def plot_alert_distribution(alerts):
    """Plot alert type distribution and severity pie chart."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    type_counts = {}
    severity_counts = {}

    for alert in alerts:
      t = alert.get('type', 'Unknown')
      s = alert.get('severity', 'Unknown')
      type_counts[t] = type_counts.get(t, 0) + 1
      severity_counts[s] = severity_counts.get(s, 0) + 1

    if type_counts:
      ax1.barh(
          list(type_counts.keys()),
          list(type_counts.values()),
          color=['#e74c3c', '#e67e22', '#3498db', '#2ecc71'][: len(type_counts)],
      )
      ax1.set_title('🚨 Alerts by Type')
      ax1.set_xlabel('Count')
    else:
      ax1.text(
          0.5,
          0.5,
          'No Alerts Recorded',
          ha='center',
          va='center',
          fontsize=12,
          color='gray',
      )

    if severity_counts:
      colors = {
          'LOW': '#2ecc71',
          'MEDIUM': '#f1c40f',
          'HIGH': '#e67e22',
          'CRITICAL': '#e74c3c',
      }
      ax2.pie(
          list(severity_counts.values()),
          labels=list(severity_counts.keys()),
          colors=[colors.get(s, '#95a5a6') for s in severity_counts.keys()],
          autopct='%1.1f%%',
          startangle=90,
      )
      ax2.set_title('⚡ Alerts by Severity')
    else:
      ax2.text(
          0.5,
          0.5,
          'No Severity Data',
          ha='center',
          va='center',
          fontsize=12,
          color='gray',
      )

    plt.tight_layout()
    output_filename = 'alert_distribution.png'
    plt.savefig(output_filename, dpi=150, bbox_inches='tight')
    plt.close()
    return output_filename

  @staticmethod
  def plot_zone_analytics(zone_counts_history):
    """Plot zone-wise people count over time."""
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    if zone_counts_history:
      for zone_name, counts in zone_counts_history.items():
        ax.plot(counts, label=zone_name, linewidth=2)
      ax.legend()
    else:
      ax.text(
          0.5,
          0.5,
          'No Zone Data Available',
          ha='center',
          va='center',
          fontsize=12,
          color='gray',
      )

    ax.set_xlabel('Time')
    ax.set_ylabel('People Count')
    ax.set_title('📍 Zone-wise Crowd Distribution')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    output_filename = 'zone_analytics.png'
    plt.savefig(output_filename, dpi=150, bbox_inches='tight')
    plt.close()
    return output_filename

  @staticmethod
  def generate_report(results):
    """Generate comprehensive Markdown analysis report."""
    if not results:
      return "⚠️ No analysis results available to generate report."

    total_frames = len(results)
    avg_crowd = np.mean([r.get('crowd_count', 0) for r in results])
    max_crowd = max([r.get('crowd_count', 0) for r in results])
    avg_clean = np.mean([r.get('cleanliness_score', 100) for r in results])
    total_criminals = sum([r.get('criminals', 0) for r in results])
    total_anomalies = sum([r.get('anomalies', 0) for r in results])

    report = f"""
# 🚂 Indian Railways AI Surveillance Report
## Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}

---

## 📊 Summary Statistics

| Metric | Value |
|--------|-------|
| Total Frames Analyzed | {total_frames} |
| Average Crowd Size | {avg_crowd:.1f} |
| Peak Crowd Size | {max_crowd} |
| Average Cleanliness | {avg_clean:.1f}% |
| Total Criminal Alerts | {total_criminals} |
| Total Anomaly Events | {total_anomalies} |

---

## 🎯 Recommendations

1. **Crowd Management**: {"⚠️ Deploy additional staff to control platform congestion" if max_crowd > 100 else "✅ Current staffing adequate"}
2. **Security**: {"🚨 Patrol frequency should be increased immediately" if total_criminals > 0 else "✅ No immediate high-priority security concerns"}
3. **Cleanliness**: {"🧹 Schedule immediate cleaning drive" if avg_clean < 70 else "✅ Cleanliness standards met"}

---
*Report generated by Indian Railways AI Surveillance System*
"""
    return report


# Instantiate Analytics Dashboard module
analytics = AnalyticsDashboard()

⏳ Processed 100/200 frames...
⏳ Processed 150/200 frames...


In [ ]:
from ultralytics import YOLO

# 1. Export YOLO11 Detection to ONNX FP16 (Fast & zero extra dependencies)
model_det = YOLO("yolo11n.pt")
model_det.export(format="onnx", half=True)

# 2. Export YOLO11 Pose to ONNX FP16
model_pose = YOLO("yolo11n-pose.pt")
model_pose.export(format="onnx", half=True)

In [ ]:
# Uninstall CPU version and install GPU version
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install -U onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/nuget/v3/index.json

In [ ]:
# ============================================
# OPTIMIZATION: Model Benchmarking & ONNX Pipeline
# ============================================
import time
import numpy as np
from ultralytics import YOLO


class QuantizedModels:
  """Utility to export models and benchmark execution speeds."""

  @staticmethod
  def export_yolo_to_onnx(model_path='yolo11n.pt'):
    """Export standard PyTorch weights to ONNX FP16 format."""
    model = YOLO(model_path)

    onnx_path = model.export(
        format='onnx',
        half=True,  # FP16 Precision
        simplify=True,  # Graph simplification
        opset=17,
    )

    print(f'✅ Exported {model_path} to ONNX FP16 -> {onnx_path}')
    return onnx_path

  @staticmethod
  def benchmark_models():
    """Compare standard PyTorch FP32 vs ONNX FP16 inference speed."""

    print('⏱️ Running Model Speed Benchmarks...\n')

    # Load models
    model_fp32 = YOLO('yolo11n.pt')
    model_fp16 = YOLO('yolo11n.onnx', task='detect')

    # Create dummy 720p frame
    dummy_frame = np.random.randint(
        0, 255, (720, 1280, 3), dtype=np.uint8
    )

    # 1. Warm-up GPU
    for _ in range(5):
      model_fp32(dummy_frame, verbose=False)
      model_fp16(dummy_frame, verbose=False)

    # 2. Benchmark PyTorch FP32
    start = time.time()
    for _ in range(50):
      model_fp32(dummy_frame, verbose=False)
    fp32_fps = 50 / (time.time() - start)

    # 3. Benchmark ONNX FP16
    start = time.time()
    for _ in range(50):
      model_fp16(dummy_frame, verbose=False)
    fp16_fps = 50 / (time.time() - start)

    # Results Output
    print('📊 Benchmark Results:')
    print(f'   FP32 (PyTorch) : {fp32_fps:.1f} FPS')
    print(f'   FP16 (ONNX)    : {fp16_fps:.1f} FPS')
    print(f'   Speedup        : {fp16_fps/fp32_fps:.2f}x faster')


# Execute Benchmark Test
QuantizedModels.benchmark_models()

In [ ]:
import shutil
from google.colab import files

# 1. Compress the project directory
shutil.make_archive(
    "railway_surveillance_ai", "zip", "/content/railway-surveillance-ai"
)

# 2. Download the zip file to your local computer
files.download("railway_surveillance_ai.zip")

RuntimeError: File size too large, try using force_zip64

In [ ]:
# ==========================================================
# 📷 REAL-TIME BROWSER WEBCAM STREAMING FOR GOOGLE COLAB
# Stream video from your laptop camera into the Colab GPU
# ==========================================================
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
import PIL.Image
import io

def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    image_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(image_np, flags=1)
    return img

def bbox_to_bytes(bbox_array):
    bbox_PIL = PIL.Image.fromarray(bbox_array)
    io_out = io.BytesIO()
    bbox_PIL.save(io_out, format='png')
    bbox_bytes = 'data:image/png;base64,' + format(str(b64encode(io_out.getvalue()), 'utf-8'))
    return bbox_bytes

def video_stream():
    js = Javascript('''
        var video;
        var container = null;
        var stream;
        var captureCanvas;
        var imgElement;
        var pendingResolve = null;
        var shutdown = false;
        
        function removeDom() {
           if (stream) stream.getVideoTracks()[0].stop();
           if (container) container.remove();
           video = null;
           container = null;
           stream = null;
           imgElement = null;
           captureCanvas = null;
           shutdown = true;
        }
        
        function onAnimationFrame() {
          if (!shutdown) {
            window.requestAnimationFrame(onAnimationFrame);
          }
          if (pendingResolve) {
            var result = "";
            if (!shutdown && video && captureCanvas) {
              captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
              result = captureCanvas.toDataURL('image/jpeg', 0.8);
            }
            var resolve = pendingResolve;
            pendingResolve = null;
            resolve(result);
          }
        }
        
        async function createDom() {
          if (container !== null) {
            return stream;
          }
          container = document.createElement('div');
          container.style.display = 'flex';
          container.style.flexDirection = 'column';
          container.style.alignItems = 'flex-start';
          container.style.margin = '10px 0';
          
          const controlBar = document.createElement('div');
          controlBar.style.marginBottom = '8px';
          
          const btn = document.createElement('button');
          btn.innerText = '⏹️ Stop Live Webcam';
          btn.style.padding = '8px 18px';
          btn.style.backgroundColor = '#ef4444';
          btn.style.color = '#ffffff';
          btn.style.border = 'none';
          btn.style.borderRadius = '6px';
          btn.style.fontSize = '14px';
          btn.style.fontWeight = 'bold';
          btn.style.cursor = 'pointer';
          btn.onclick = removeDom;
          controlBar.appendChild(btn);
          container.appendChild(controlBar);
          
          // Relative wrapper so video and AI overlay stack exactly on top of each other
          const videoWrapper = document.createElement('div');
          videoWrapper.style.position = 'relative';
          videoWrapper.style.width = '640px';
          videoWrapper.style.height = '480px';
          videoWrapper.style.borderRadius = '10px';
          videoWrapper.style.overflow = 'hidden';
          videoWrapper.style.border = '2px solid #22c55e';
          videoWrapper.style.boxShadow = '0 6px 20px rgba(0,0,0,0.4)';
          
          video = document.createElement('video');
          video.style.position = 'absolute';
          video.style.top = '0';
          video.style.left = '0';
          video.style.width = '640px';
          video.style.height = '480px';
          video.style.objectFit = 'cover';
          video.setAttribute('playsinline', '');
          
          stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480, facingMode: "user"}});
          videoWrapper.appendChild(video);
          
          imgElement = document.createElement('img');
          imgElement.style.position = 'absolute';
          imgElement.style.top = '0';
          imgElement.style.left = '0';
          imgElement.style.width = '640px';
          imgElement.style.height = '480px';
          imgElement.style.objectFit = 'cover';
          imgElement.style.zIndex = '10';
          imgElement.style.pointerEvents = 'none';
          videoWrapper.appendChild(imgElement);
          
          container.appendChild(videoWrapper);
          document.body.appendChild(container);
          
          video.srcObject = stream;
          await video.play();
          
          captureCanvas = document.createElement('canvas');
          captureCanvas.width = 640;
          captureCanvas.height = 480;
          window.requestAnimationFrame(onAnimationFrame);
          return stream;
        }
        
        async function stream_frame(label, imgData) {
          if (shutdown) {
            removeDom();
            shutdown = true;
            return '';
          }
          stream = await createDom();
          if (label != "" && imgElement) {
            imgElement.src = "data:image/png;base64," + imgData;
          }
          var result = await new Promise(function(resolve, reject) {
            pendingResolve = resolve;
          });
          shutdown = false;
          return result;
        }
        ''')
    display(js)

def stream_colab_ai(max_frames=300):
    """Runs live camera stream from laptop browser directly into Colab GPU pipeline."""
    video_stream()
    label_html = 'Live Colab Webcam'
    bbox = ''
    count = 0
    print("🔴 Live Webcam stream started on Colab! AI overlays are rendered directly on top of your video.")
    
    while count < max_frames:
        js_reply = eval_js('stream_frame("{}", "{}")'.format(label_html, bbox))
        if not js_reply:
            break
        
        frame = js_to_image(js_reply)
        results = pipeline.process_frame(frame)
        
        # Create transparent overlay for annotations
        annotated = results['frame']
        bbox_array = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGBA)
        bbox_bytes = bbox_to_bytes(bbox_array)
        bbox = bbox_bytes.split(',')[1]
        count += 1

# To run live webcam directly in Colab cell:
# stream_colab_ai(max_frames=300)
